# 02 - Pipeline Medallion

Este notebook implementa o pipeline do projeto Olist seguindo a arquitetura Landing Zone → Bronze → Silver → Gold, com camada de quarentena e histórico de clientes.

Decisões principais:
- A Bronze lê os CSVs da camada `01_raw`, mantendo os dados brutos, metadados de ingestão, arquivo de origem e coluna de resgate.
- A Silver concentra tratamento, validações e relacionamentos entre clientes, pedidos, itens, pagamentos, produtos e geolocalização.
- A regra obrigatória de qualidade é aplicada na Silver: clientes com pelo menos dois pedidos seguem como válidos; os demais vão para quarentena.
- A tabela de clientes recebe estrutura histórica, pois mudanças em CEP, cidade, estado e localização impactam análises de comportamento de compra.
- A Gold consolida pagamentos por cliente, método de pagamento, dia e mês da compra, com as métricas exigidas no enunciado.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## 1. Configurações iniciais

In [0]:
catalog = "tp1_olist"

raw_path = "/Volumes/tp1_olist/01_raw/olist_raw/"
bad_records_path = "/Volumes/tp1_olist/01_raw/olist_raw/_bad_records/"

bronze_schema = "02_bronze"
silver_schema = "03_silver"
gold_schema = "04_gold"
quarantine_schema = "05_quarantine"

In [0]:
def table_name(schema_name, table_name):
    return f"{catalog}.`{schema_name}`.{table_name}"

## 2. Camada Bronze

Nesta camada os arquivos CSV da `01_raw` são convertidos em Delta Tables, preservando os dados brutos e adicionando metadados de ingestão.

In [0]:
# Arquivos utilizados no pipeline

csv_files = {
    "customers_bronze": "olist_customers_dataset.csv",
    "orders_bronze": "olist_orders_dataset.csv",
    "items_bronze": "olist_order_items_dataset.csv",
    "payments_bronze": "olist_order_payments_dataset.csv",
    "products_bronze": "olist_products_dataset.csv",
    "geolocation_bronze": "olist_geolocation_dataset.csv",
    "categories_bronze": "product_category_name_translation.csv",
    "reviews_bronze": "olist_order_reviews_dataset.csv",
    "sellers_bronze": "olist_sellers_dataset.csv"
}

In [0]:
display(
    dbutils.fs.ls(raw_path)
)

In [0]:
# Validação dos arquivos esperados na camada Raw

existing_files = [file.name for file in dbutils.fs.ls(raw_path)]

missing_files = [
    file_name
    for file_name in csv_files.values()
    if file_name not in existing_files
]

if missing_files:
    raise Exception(f"Arquivos não encontrados: {missing_files}")

print("Todos os arquivos esperados foram encontrados.")

In [0]:
# Função padrão de ingestão da Bronze

def read_csv_with_rescue(file_name, bronze_table_name):

    file_path = f"{raw_path}{file_name}"

    inferred_schema = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
        .schema
    )

    schema_with_rescue = StructType(
        inferred_schema.fields + [
            StructField("_rescued_data", StringType(), True)
        ]
    )

    return (
        spark.read
        .format("csv")
        .option("header", True)
        .option("columnNameOfCorruptRecord", "_rescued_data")
        .option("badRecordsPath", f"{bad_records_path}{bronze_table_name}")
        .schema(schema_with_rescue)
        .load(file_path)
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
    )

In [0]:
# Criação das tabelas Bronze

for bronze_table_name, file_name in csv_files.items():

    df = read_csv_with_rescue(
        file_name,
        bronze_table_name
    )

    (
        df.write

        .format("delta")

        .mode("overwrite")

        .option("overwriteSchema", True)

        .saveAsTable(
            table_name(
                bronze_schema,
                bronze_table_name
            )
        )
    )

    print(
        f"Tabela criada: {bronze_table_name}"
    )

## 3. Camada Silver

Nesta camada são aplicadas validações, padronizações e relacionamentos necessários para consumo analítico.

In [0]:
# Valores válidos utilizados nas validações

VALID_UF = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

VALID_STATUS = [
    "created", "approved", "invoiced", "processing",
    "shipped", "delivered", "canceled", "unavailable"
]

VALID_PAYMENT_TYPES = [
    "credit_card",
    "boleto",
    "voucher",
    "debit_card",
    "not_defined"
]

### 3.1 Produtos

In [0]:
products_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "products_bronze"
        )
    )

    .withColumn(
        "product_id",
        trim(col("product_id"))
    )

    .withColumn(

        "product_category_name",

        coalesce(
            lower(
                trim(
                    col("product_category_name")
                )
            ),
            lit("unknown")
        )
    )

    .withColumn(
        "product_weight_g",
        col("product_weight_g").cast("double")
    )

    .withColumn(
        "product_length_cm",
        col("product_length_cm").cast("double")
    )

    .withColumn(
        "product_height_cm",
        col("product_height_cm").cast("double")
    )

    .withColumn(
        "product_width_cm",
        col("product_width_cm").cast("double")
    )

    .filter(col("product_id").isNotNull())

    .filter(col("product_weight_g") > 0)

    .filter(col("product_length_cm") > 0)

    .filter(col("product_height_cm") > 0)

    .filter(col("product_width_cm") > 0)

    .dropDuplicates(["product_id"])
)

In [0]:
(
    products_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "products_silver"
        )
    )
)

### 3.2 Itens dos pedidos

In [0]:
# Mantém apenas itens relacionados a produtos válidos

items_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "items_bronze"
        )
    )

    .withColumn(
        "order_id",
        trim(col("order_id"))
    )

    .withColumn(
        "product_id",
        trim(col("product_id"))
    )

    .withColumn(
        "seller_id",
        trim(col("seller_id"))
    )

    .withColumn(
        "order_item_id",
        col("order_item_id").cast("int")
    )

    .withColumn(
        "price",
        col("price").cast("double")
    )

    .withColumn(
        "freight_value",
        col("freight_value").cast("double")
    )

    .filter(col("order_id").isNotNull())

    .filter(col("product_id").isNotNull())

    .filter(col("seller_id").isNotNull())

    .filter(col("order_item_id") > 0)

    .filter(col("price") > 0)

    .filter(col("freight_value") >= 0)

    .dropDuplicates([
        "order_id",
        "order_item_id"
    ])

    .join(

        spark.table(
            table_name(
                silver_schema,
                "products_silver"
            )
        ).select("product_id"),

        "product_id",

        "inner"
    )
)

In [0]:
(
    items_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "items_silver"
        )
    )
)

### 3.3 Clientes

In [0]:
# Coordenadas médias por CEP

geo_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "geolocation_bronze"
        )
    )

    .withColumn(

        "geolocation_zip_code_prefix",

        lpad(
            col(
                "geolocation_zip_code_prefix"
            ).cast("string"),
            5,
            "0"
        )
    )

    .groupBy(
        "geolocation_zip_code_prefix"
    )

    .agg(

        avg(
            "geolocation_lat"
        ).alias("latitude"),

        avg(
            "geolocation_lng"
        ).alias("longitude")
    )
)

In [0]:
customers_current_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "customers_bronze"
        )
    )

    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )

    .withColumn(
        "customer_unique_id",
        trim(col("customer_unique_id"))
    )

    .withColumn(

        "customer_zip_code_prefix",

        lpad(
            col(
                "customer_zip_code_prefix"
            ).cast("string"),
            5,
            "0"
        )
    )

    .withColumn(
        "customer_city",
        lower(
            trim(
                col("customer_city")
            )
        )
    )

    .withColumn(
        "customer_state",
        upper(
            trim(
                col("customer_state")
            )
        )
    )

    .filter(
        col("customer_id").isNotNull()
    )

    .filter(
        col("customer_unique_id").isNotNull()
    )

    .filter(
        col("customer_zip_code_prefix").isNotNull()
    )

    .filter(
        col("customer_city").isNotNull()
    )

    .filter(
        col("customer_state").isin(VALID_UF)
    )

    .dropDuplicates([
    "customer_id"
])

    .join(

        geo_silver,

        col(
            "customer_zip_code_prefix"
        )

        ==

        col(
            "geolocation_zip_code_prefix"
        ),

        "left"
    )

    .select(
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
        "latitude",
        "longitude",
        "ingestion_timestamp"
    )
)

In [0]:
(
    customers_current_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "customers_silver"
        )
    )
)

### Histórico de clientes

In [0]:
# Estrutura histórica simplificada para rastrear alterações cadastrais

customers_history_silver = (

    customers_current_silver

    .withColumn(
        "valid_from",
        col("ingestion_timestamp")
    )

    .withColumn(
        "valid_to",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        lit(True)
    )
)

In [0]:
(
    customers_history_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "customers_history_silver"
        )
    )
)

### 3.4 Pedidos

In [0]:
# Quantidade de itens por pedido

item_count = (

    spark.table(
        table_name(
            silver_schema,
            "items_silver"
        )
    )

    .groupBy("order_id")

    .agg(
        count("*").alias("total_items")
    )
)

In [0]:
orders_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "orders_bronze"
        )
    )

    .withColumn(
        "order_id",
        trim(col("order_id"))
    )

    .withColumn(
        "customer_id",
        trim(col("customer_id"))
    )

    .withColumn(
        "order_status",
        lower(
            trim(
                col("order_status")
            )
        )
    )

    .withColumn(
        "order_purchase_timestamp",
        to_timestamp(
            "order_purchase_timestamp"
        )
    )

    .withColumn(
        "order_approved_at",
        to_timestamp(
            "order_approved_at"
        )
    )

    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp(
            "order_delivered_carrier_date"
        )
    )

    .withColumn(
        "order_delivered_customer_date",
        to_timestamp(
            "order_delivered_customer_date"
        )
    )

    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp(
            "order_estimated_delivery_date"
        )
    )

    .filter(col("order_id").isNotNull())

    .filter(col("customer_id").isNotNull())

    .filter(
        col("order_status").isin(
            VALID_STATUS
        )
    )

    .filter(
        col(
            "order_purchase_timestamp"
        ).isNotNull()
    )

    .filter(
        col(
            "order_estimated_delivery_date"
        ).isNotNull()
    )

    # Evita datas futuras
    .filter(
        col(
            "order_purchase_timestamp"
        ) <= current_timestamp()
    )

    # Consistência temporal
    .filter(

        (
            col("order_approved_at").isNull()
        )

        |

        (
            col(
                "order_purchase_timestamp"
            )

            <=

            col("order_approved_at")
        )
    )

    .filter(

        (
            col(
                "order_delivered_carrier_date"
            ).isNull()
        )

        |

        (
            col(
                "order_approved_at"
            ).isNull()
        )

        |

        (
            col(
                "order_approved_at"
            )

            <=

            col(
                "order_delivered_carrier_date"
            )
        )
    )

    .filter(

        (
            col(
                "order_delivered_customer_date"
            ).isNull()
        )

        |

        (
            col(
                "order_delivered_carrier_date"
            ).isNull()
        )

        |

        (
            col(
                "order_delivered_carrier_date"
            )

            <=

            col(
                "order_delivered_customer_date"
            )
        )
    )

    # Pedidos entregues precisam ter data de entrega
    .filter(

        (
            col("order_status") != "delivered"
        )

        |

        (
            col(
                "order_delivered_customer_date"
            ).isNotNull()
        )
    )

    # Mantém apenas pedidos relacionados a clientes válidos
    .join(

        spark.table(
            table_name(
                silver_schema,
                "customers_silver"
            )
        ).select("customer_id"),

        "customer_id",

        "inner"
    )

    .join(
        item_count,
        "order_id",
        "left"
    )

    .withColumn(
        "total_items",
        coalesce(
            col("total_items"),
            lit(0)
        )
    )

    # Tempo de aprovação
    .withColumn(

        "approval_hours",

        (

            unix_timestamp(
                "order_approved_at"
            )

            -

            unix_timestamp(
                "order_purchase_timestamp"
            )

        ) / 3600
    )

    # Tempo total de entrega
    .withColumn(

        "delivery_days",

        datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )

    # Atraso de entrega
    .withColumn(

        "delivery_delay_days",

        datediff(
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        )
    )

    .dropDuplicates(["order_id"])

    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "approval_hours",
        "delivery_days",
        "delivery_delay_days",
        "total_items"
    )
)

In [0]:
(
    orders_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "orders_silver"
        )
    )
)

### 3.5 Pagamentos

In [0]:
payments_silver = (

    spark.table(
        table_name(
            bronze_schema,
            "payments_bronze"
        )
    )

    .withColumn(
        "order_id",
        trim(col("order_id"))
    )

    .withColumn(
        "payment_type",
        lower(
            trim(
                col("payment_type")
            )
        )
    )

    .withColumn(
        "payment_sequential",
        col(
            "payment_sequential"
        ).cast("int")
    )

    .withColumn(
        "payment_installments",
        col(
            "payment_installments"
        ).cast("int")
    )

    .withColumn(
        "payment_value",
        col(
            "payment_value"
        ).cast("double")
    )

    .filter(
        col("order_id").isNotNull()
    )

    .filter(
        col(
            "payment_type"
        ).isin(
            VALID_PAYMENT_TYPES
        )
    )

    .filter(
        col("payment_value") > 0
    )

    .filter(
        col(
            "payment_installments"
        ) >= 1
    )

    .filter(
        col(
            "payment_sequential"
        ) >= 1
    )

    # Mantém apenas pagamentos relacionados a pedidos válidos
    .join(

        spark.table(
            table_name(
                silver_schema,
                "orders_silver"
            )
        ).select("order_id"),

        "order_id",

        "inner"
    )

    .select(
        "order_id",
        "payment_type",
        "payment_sequential",
        "payment_installments",
        "payment_value"
    )
)

In [0]:
(
    payments_silver.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            silver_schema,
            "payments_silver"
        )
    )
)

## 4. Qualidade de dados e quarentena

Clientes com menos de dois pedidos são enviados para a camada `05_quarantine`.

In [0]:
# Remove versões antigas para evitar inconsistência
spark.sql(f"DROP TABLE IF EXISTS {table_name(silver_schema, 'customer_orders_validation')}")
spark.sql(f"DROP TABLE IF EXISTS {table_name(silver_schema, 'valid_customers')}")
spark.sql(f"DROP TABLE IF EXISTS {table_name(quarantine_schema, 'customers_quarantine')}")

In [0]:
customer_orders_validation = (
    spark.table(table_name(silver_schema, "orders_silver"))
    .alias("o")
    .join(
        spark.table(table_name(silver_schema, "customers_silver"))
        .select("customer_id", "customer_unique_id")
        .alias("c"),
        "customer_id",
        "inner"
    )
    .groupBy("customer_unique_id")
    .agg(
        countDistinct("order_id").alias("total_orders")
    )
)

(
    customer_orders_validation.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_name(silver_schema, "customer_orders_validation"))
)

In [0]:
valid_customers = (
    spark.table(table_name(silver_schema, "customer_orders_validation"))
    .filter(col("total_orders") >= 2)
)

(
    valid_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_name(silver_schema, "valid_customers"))
)

In [0]:
customers_quarantine = (
    spark.table(table_name(silver_schema, "customer_orders_validation"))
    .filter(col("total_orders") < 2)
    .withColumn("quarantine_reason", lit("Cliente com menos de dois pedidos"))
    .withColumn("quarantine_timestamp", current_timestamp())
)

(
    customers_quarantine.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_name(quarantine_schema, "customers_quarantine"))
)

In [0]:
validation_count = spark.table(table_name(silver_schema, "customer_orders_validation")).count()
valid_count = spark.table(table_name(silver_schema, "valid_customers")).count()
quarantine_count = spark.table(table_name(quarantine_schema, "customers_quarantine")).count()

print(f"Validation: {validation_count}")
print(f"Valid: {valid_count}")
print(f"Quarantine: {quarantine_count}")
print(f"Valid + Quarantine: {valid_count + quarantine_count}")

## 5. Camada Gold

Tabela analítica consolidada por cliente, método de pagamento, mês e dia da compra.

A métrica `avg_orders` foi mantida para atender ao enunciado. Como a tabela já está agregada por cliente, método de pagamento, mês e dia da compra, a média de pedidos nessa granularidade equivale à quantidade de pedidos do próprio grupo.

In [0]:
base_order_payment = (

    spark.table(
        table_name(
            silver_schema,
            "orders_silver"
        )
    ).alias("o")

    .join(

        spark.table(
            table_name(
                silver_schema,
                "payments_silver"
            )
        ).alias("p"),

        "order_id",

        "inner"
    )

    .withColumn(
        "purchase_month",
        month(
            col(
                "order_purchase_timestamp"
            )
        )
    )

    .withColumn(
        "purchase_day",
        dayofmonth(
            col(
                "order_purchase_timestamp"
            )
        )
    )

    .select(
        "customer_id",
        "payment_type",
        "purchase_month",
        "purchase_day",
        "order_id",
        "total_items",
        "payment_value",
        "payment_installments"
    )
)

In [0]:
# Agregação no nível do pedido

order_payment_aggregated = (

    base_order_payment

    .groupBy(
        "customer_id",
        "payment_type",
        "purchase_month",
        "purchase_day",
        "order_id"
    )

    .agg(

        first(
            "total_items"
        ).alias(
            "order_total_items"
        ),

        sum(
            "payment_value"
        ).alias(
            "order_total_paid"
        ),

        avg(
            "payment_installments"
        ).alias(
            "order_avg_installments"
        )
    )
)

In [0]:
customer_payments_gold = (

    order_payment_aggregated

    .groupBy(
        "customer_id",
        "payment_type",
        "purchase_month",
        "purchase_day"
    )

    .agg(

        countDistinct(
            "order_id"
        ).alias(
            "total_orders"
        ),

        sum(
            "order_total_items"
        ).alias(
            "total_items"
        ),

        sum(
            "order_total_paid"
        ).alias(
            "total_paid"
        ),

        avg(
            "order_total_items"
        ).alias(
            "avg_items"
        ),

        avg(
            "order_total_paid"
        ).alias(
            "avg_payment_value"
        ),

        avg(
            "order_avg_installments"
        ).alias(
            "avg_installments"
        )
    )

    .withColumn(
        "avg_orders",
        col("total_orders").cast("double")
    )

    .select(
        "customer_id",
        "payment_type",
        "purchase_month",
        "purchase_day",
        "total_orders",
        "total_items",
        "total_paid",
        "avg_orders",
        "avg_items",
        "avg_payment_value",
        "avg_installments"
    )
)

In [0]:
(
    customer_payments_gold.write

    .format("delta")

    .mode("overwrite")

    .option("overwriteSchema", True)

    .saveAsTable(
        table_name(
            gold_schema,
            "customer_payments_gold"
        )
    )
)

## 6. Evidências de execução

Nesta etapa são geradas as principais evidências solicitadas no enunciado: arquivos na Landing Zone, tabelas criadas no Unity Catalog, contagens por tabela, amostras das camadas Bronze, Silver e Gold, além do resultado da regra de qualidade e da quarentena.

In [0]:
# Evidência dos arquivos CSV armazenados na Landing Zone

display(
    dbutils.fs.ls(raw_path)
)

In [0]:
# Evidência das tabelas criadas no Unity Catalog

schemas_to_check = [
    bronze_schema,
    silver_schema,
    gold_schema,
    quarantine_schema
]

for schema_name in schemas_to_check:
    print(f"\nTabelas no schema {catalog}.{schema_name}:")
    
    tables = spark.sql(f"SHOW TABLES IN {catalog}.`{schema_name}`")
    display(tables)

In [0]:
display(

    spark.table(
        table_name(
            gold_schema,
            "customer_payments_gold"
        )
    ).limit(10)
)

In [0]:
# Contagem de registros por tabela

tables_to_validate = [
    table_name(bronze_schema, "customers_bronze"),
    table_name(bronze_schema, "orders_bronze"),
    table_name(bronze_schema, "items_bronze"),
    table_name(bronze_schema, "payments_bronze"),
    table_name(bronze_schema, "products_bronze"),
    table_name(bronze_schema, "geolocation_bronze"),
    table_name(bronze_schema, "categories_bronze"),
    table_name(silver_schema, "customers_silver"),
    table_name(silver_schema, "orders_silver"),
    table_name(silver_schema, "payments_silver"),
    table_name(quarantine_schema, "customers_quarantine"),
    table_name(gold_schema, "customer_payments_gold")
]

for table in tables_to_validate:
    total_rows = spark.table(table).count()
    print(f"{table}: {total_rows} registros")

In [0]:
# Amostra da camada Bronze

display(
    spark.table(
        table_name(bronze_schema, "orders_bronze")
    ).limit(10)
)

In [0]:
# Amostra da camada Silver

display(
    spark.table(
        table_name(silver_schema, "orders_silver")
    ).limit(10)
)

In [0]:
# Amostra da camada Gold

display(
    spark.table(
        table_name(gold_schema, "customer_payments_gold")
    ).limit(10)
)

In [0]:
# Resultado da regra de qualidade

quality_result = (
    spark.table(
        table_name(silver_schema, "customer_orders_validation")
    )
    .withColumn(
        "quality_status",
        when(col("total_orders") >= 2, lit("valid"))
        .otherwise(lit("quarantine"))
    )
    .groupBy("quality_status")
    .agg(count("*").alias("total_customers"))
)

display(quality_result)

In [0]:
# Quantidade de registros enviados para quarentena

quarantine_count = spark.table(
    table_name(quarantine_schema, "customers_quarantine")
).count()

print(f"Clientes enviados para quarentena: {quarantine_count}")

In [0]:
# Amostra dos registros enviados para quarentena

display(
    spark.table(
        table_name(quarantine_schema, "customers_quarantine")
    ).limit(10)
)

## 7. Conclusão

O pipeline implementou as camadas Bronze, Silver, Quarantine e Gold, preservando os dados brutos, aplicando validações e disponibilizando uma tabela analítica consolidada para consumo de negócio.